In [8]:
import duckdb

In [9]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [10]:
df = con.execute("""
                SELECT *
                FROM (
                SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row 
                FROM bronze_z0019
                WHERE data_ingestao >= '2026-05-19'
            ) WHERE row = 1
            """).fetchdf()
df

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-05-19 17:22:32.384488,1
1,10002,MARTELO,BT50,100,500,z0019_1.csv,2026-05-19 17:22:32.384488,1
2,10004,SERRA,BT50,100,200,z0019_2.csv,2026-05-19 17:32:51.516267,1
3,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-05-19 17:32:51.516267,1
4,10003,PREGO,BT10,100,60,z0019_2.csv,2026-05-19 17:32:51.516267,1


In [11]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'row'])
df_final = df_final.rename(columns={"NATBR":"id"})
df_final = df_final.rename(columns={"MAKTX":"nm_produto"})
df_final = df_final.rename(columns={"WERKS":"id_catogoria"})
df_final = df_final.rename(columns={"MAINS":"id_fornecedor"})
df_final = df_final.rename(columns={"LABST":"vl_preco"})
df_final

,id,nm_produto,id_catogoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,500
2,10004,SERRA,BT50,100,200
3,10005,MACHADO,BT50,100,100
4,10003,PREGO,BT10,100,60


In [19]:
df2 = df_final
df2 = df2.astype(
    {
        'id': int,
        'nm_produto': str,
        'id_categoria': str,
        'id_fornecedor': int,
        'vl_preco': float
    }
)

In [32]:
con.execute("""
CREATE TABLE IF NOT EXISTS produtos (
            id BIGINT,
            nm_produto TEXT,
            id_categoria TEXT,
            id_fornecedor BIGINT,
            vl_preco FLOAT    
            )

""")

In [33]:
con.execute("INSERT INTO produtos SELECT * FROM df2")

In [34]:
df_resultado = con.execute("SELECT * FROM produtos").fetch_df()
df_resultado

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,500.0
2,10004,SERRA,BT50,100,200.0
3,10005,MACHADO,BT50,100,100.0
4,10003,PREGO,BT10,100,60.0


In [35]:
con.close()